<a href="https://colab.research.google.com/github/ChaitanReddy/Gen-AI-Assignment/blob/main/Task_4_BERT_Fine_Tuning_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Assignment – BERT Fine-Tuning

Text Classification using BERT (bert-base-uncased) with preprocessing, training, evaluation, and experiments.


In [ ]:
# Install dependencies
!pip install transformers datasets scikit-learn torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [ ]:
# Load dataset (IMDB)
dataset = load_dataset('imdb')
train_data = dataset['train']
test_data = dataset['test']


In [ ]:
# Tokenization
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data.set_format(type='torch', columns=['input_ids','attention_mask','label'])
test_data.set_format(type='torch', columns=['input_ids','attention_mask','label'])


In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
optimizer = AdamW(model.parameters(), lr=2e-5)


In [ ]:
# Training (light version for demo)
model.train()

for epoch in range(1):
    for batch in train_data.select(range(200)):
        outputs = model(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0),
            labels=batch['label'].unsqueeze(0)
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


In [ ]:
# Evaluation
model.eval()

preds, labels = [], []

# Sample a balanced subset for evaluation
from collections import Counter

test_labels = [example['label'] for example in test_data.select(range(1000))] # Temporarily use a larger range to check distribution
label_counts = Counter(test_labels)

# Determine how many samples to take from each class for a balanced subset of 200
samples_per_class = 100 # For a total of 200 samples

balanced_test_indices = []
class_0_indices = [i for i, label in enumerate(test_labels) if label == 0]
class_1_indices = [i for i, label in enumerate(test_labels) if label == 1]

# Take `samples_per_class` from each class, ensuring we don't exceed available samples
balanced_test_indices.extend(class_0_indices[:samples_per_class])
balanced_test_indices.extend(class_1_indices[:samples_per_class])

# Shuffle the indices to mix classes
import random
random.shuffle(balanced_test_indices)


with torch.no_grad():
    # Iterate over the balanced subset
    for idx in balanced_test_indices:
        batch = test_data[idx]
        outputs = model(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0)
        )
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()
        preds.append(pred)
        labels.append(batch['label'].item())


acc = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("Confusion Matrix:")
print(confusion_matrix(labels, preds))

In [ ]:
results = {}
results['Full Fine-tune (Light)'] = {
    'Accuracy': acc,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1
}

print("Initial Full Fine-tune (Light) Results:")
for metric, value in results['Full Fine-tune (Light)'].items():
    print(f"  {metric}: {value:.4f}")

### Experiment 1: Freeze BERT Layers


In [ ]:
# Load a fresh model for this experiment
model_frozen = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Freeze all parameters in the BERT encoder
for param in model_frozen.bert.parameters():
    param.requires_grad = False

# Ensure the classifier layers are trainable
for param in model_frozen.classifier.parameters():
    param.requires_grad = True

optimizer_frozen = AdamW(model_frozen.parameters(), lr=2e-5)

# Training Loop (light version)
model_frozen.train()
print("\n--- Training with Frozen BERT Layers ---")
for epoch in range(1):
    for i, batch in enumerate(train_data.select(range(200))):
        optimizer_frozen.zero_grad()
        outputs = model_frozen(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0),
            labels=batch['label'].unsqueeze(0)
        )
        loss = outputs.loss
        loss.backward()
        optimizer_frozen.step()
        if i % 50 == 0:
            print(f"  Batch {i}/{len(train_data.select(range(200)))} Loss: {loss.item():.4f}")

# Evaluation Loop
model_frozen.eval()
preds_frozen, labels_frozen = [], []

print("\n--- Evaluating Frozen BERT Layers ---")
with torch.no_grad():
    # Iterate over the balanced subset for evaluation (created above)
    for idx in balanced_test_indices:
        batch = test_data[idx]
        outputs = model_frozen(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0)
        )
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()
        preds_frozen.append(pred)
        labels_frozen.append(batch['label'].item())

acc_frozen = accuracy_score(labels_frozen, preds_frozen)
precision_frozen, recall_frozen, f1_frozen, _ = precision_recall_fscore_support(labels_frozen, preds_frozen, average='binary', zero_division=0)

results['Frozen BERT Layers'] = {
    'Accuracy': acc_frozen,
    'Precision': precision_frozen,
    'Recall': recall_frozen,
    'F1 Score': f1_frozen
}

print("\nFrozen BERT Layers Results:")
for metric, value in results['Frozen BERT Layers'].items():
    print(f"  {metric}: {value:.4f}")

### Experiment 2: Fine-tune Last 2 Layers



In [ ]:
# Load a fresh model for this experiment
model_last_2 = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Freeze all parameters in the BERT encoder initially
for param in model_last_2.bert.parameters():
    param.requires_grad = False

# Unfreeze the last two encoder layers
# BERT's encoder layers are typically accessed via model.bert.encoder.layer
# Python list indexing allows access to the last elements using negative indices
for i in range(1, 3): # -1 and -2 for the last two layers
    for param in model_last_2.bert.encoder.layer[-i].parameters():
        param.requires_grad = True

# Ensure the classifier layers are trainable
for param in model_last_2.classifier.parameters():
    param.requires_grad = True

optimizer_last_2 = AdamW(model_last_2.parameters(), lr=2e-5)

# Training Loop (light version)
model_last_2.train()
print("\n--- Training with Fine-tuned Last 2 BERT Layers ---")
for epoch in range(1):
    for i, batch in enumerate(train_data.select(range(200))):
        optimizer_last_2.zero_grad()
        outputs = model_last_2(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0),
            labels=batch['label'].unsqueeze(0)
        )
        loss = outputs.loss
        loss.backward()
        optimizer_last_2.step()
        if i % 50 == 0:
            print(f"  Batch {i}/{len(train_data.select(range(200)))} Loss: {loss.item():.4f}")

# Evaluation Loop
model_last_2.eval()
preds_last_2, labels_last_2 = [], []

print("\n--- Evaluating Fine-tuned Last 2 BERT Layers ---")
with torch.no_grad():
    # Iterate over the balanced subset for evaluation (created above)
    for idx in balanced_test_indices:
        batch = test_data[idx]
        outputs = model_last_2(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0)
        )
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()
        preds_last_2.append(pred)
        labels_last_2.append(batch['label'].item())

acc_last_2 = accuracy_score(labels_last_2, preds_last_2)
precision_last_2, recall_last_2, f1_last_2, _ = precision_recall_fscore_support(labels_last_2, preds_last_2, average='binary', zero_division=0)

results['Fine-tune Last 2 Layers'] = {
    'Accuracy': acc_last_2,
    'Precision': precision_last_2,
    'Recall': recall_last_2,
    'F1 Score': f1_last_2
}

print("\nFine-tune Last 2 Layers Results:")
for metric, value in results['Fine-tune Last 2 Layers'].items():
    print(f"  {metric}: {value:.4f}")

### Experiment 3: Compare Performance Metrics



In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).T
print("\n--- Comparison of Experiment Results ---")
display(results_df.round(4))